# Assignment 7
1. Import the data located at this link. It has information on people infected with dengue at the district level for 2015 to 2021.
2. Generate ubigeo for Departments and Provinces taking the first two and four numbers. Hint: Use this code.
3. Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use      this shapefile.
4. Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use      this shapefile. For this task, you will have to aggregate shapefiles at the province level.
5. Use geopandas to plot the number of cases by the department for all the years using subplots. Every subplot for each year. Do not forget to indicate    the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at the department level.
6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical     legend with 5 bins. Do not forget to indicate the color of NA values. Use this shapefile. For this task, you will have to aggregate shapefiles at       the department level. Hint: Use Semana variable to group by quarters.a

Group 3:
Claudia Córdova Yamauchi, Mauricio Flores Jiménez, Reynaldo Padilla Milla, Fátima Trujillo Quiñe

# Cargamos las librerías necesarias

In [ ]:
#!pip install mapclassify

In [ ]:
#Required Libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pandas import Series, DataFrame
import chardet

import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString
import folium
from folium import Marker, GeoJson
from folium.plugins import MarkerCluster, HeatMap
import mapclassify

#1. Importar la base de datos

Cargamos la base de datos, nos aseguramos de que la configuranción es la recomendada y hacemos una inspección rápida.

In [ ]:
charenc = chardet.detect(open('../../_data/data_dengue_peru.csv', 'rb').read())['encoding']
print(charenc)
print('\n')
dengue = pd.read_csv( '../../_data/data_dengue_peru.csv', encoding = charenc)
print('Dimensiones del df:', dengue.shape)
print('\n')
print(dengue.dtypes)

In [ ]:
dengue.head(5)

In [ ]:
cols_1 = ['Eventos o daños', 'Departamento', 'Distrito', 'Provincia']
dengue[cols_1] = dengue[cols_1].astype('string')
dengue['Semana'] = dengue['Semana'].astype('Int64')
print(dengue.dtypes)
print('\n')
#Por el tipo de data, el identificador se repetirá varias veces (hay datos por año y semana). Es normal que no sea único.
print(dengue['Ubigeo'].is_unique)
print('\n')
#Verificamos si la columna Casos tiene valores con coma (p. ej., 1,000 en vez de 1000).
#Si True, al menos uno tiene coma.
ValorConComa = dengue['Casos'].str.contains(',').any()
print(ValorConComa)

In [ ]:
#Pasamos la columna 'Ubigeo' (ID del distrito) a string. Si está como integer o Int64, se eliminan los 0 a la izquierda.
#El ID debe tener 6 dígitos. Si solo hay 5, le añadimos un 0 al inicio. 
dengue['Ubigeo'] = dengue['Ubigeo'].apply(lambda row: str(row).zfill(6))
dengue.head(5)

In [ ]:
# Recodificamos los valores de la columna 'Casos'.
# Primero, aplicamos una función lambda a cada valor (fila) de la columna 'Casos'.
# La función convierte cada valor a string y elimina la coma (si la tiene), pero solo si el valor no es NaN.
# Si el valor es NaN (pd.notna(row) es False), lo dejamos sin cambios.
# Después, convertimos los valores a 'float' para que Pandas permita la conversión a 'Int64'.
# Finalmente, convertimos los valores a 'Int64', lo cual es adecuado para un conteo de casos.
# Usamos 'Int64' en lugar de 'int' para permitir que los valores NaN se mantengan sin alterarse.
dengue['Casos'] = dengue['Casos'].apply(lambda row: str(row).replace(',', '') if pd.notna(row) else row).astype(float).astype('Int64')
dengue.head(5)

#2. Creamos Ubigeo para los departamentos y las provincias

In [ ]:
#Creamos la columna 'Ubigeo_Departamento' extrayendo los dos primeros caracteres de 'Ubigeo'.
dengue['Ubigeo_Departamento'] = dengue['Ubigeo'].str[0:2]
#Creamos la columna 'Ubigeo_Provincia' extrayendo los cuatro primeros caracteres de 'Ubigeo'.
dengue['Ubigeo_Provincia'] = dengue['Ubigeo'].str[0:4]
dengue.head(5)

#3. Casos anuales de dengue a nivel distrital en el 2021


### Preparamos los datos para realizar la primera figura. Estos pasos también servirán para el resto de gráficos. 

In [ ]:
#Cargamos los datos del shapefile para obtener los polígonos (mapas) de los distritos
distritos_shp = gpd.read_file('../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')
#Verificamos si el archivo tiene información sobre la proyección. Debe ser EPSG: 4326 (para WGS-84).
print(distritos_shp.crs), print('\n')
print(distritos_shp.shape, '\n')
distritos_shp.head(5)

In [ ]:
print('¿Los valores de UBIGEO son únicos?', distritos_shp['UBIGEO'].is_unique, '. Hay:', distritos_shp['UBIGEO'].unique().size)
#Al 2021, el Perú tenía 1874 distritos.
print(distritos_shp['UBIGEO'].dtype)

In [ ]:
mapas = distritos_shp[['CCDD', 'IDPROV', 'UBIGEO', 'geometry']].copy()
mapas.rename(columns = {'CCDD': 'Ubigeo_Departamento', 'IDPROV': 'Ubigeo_Provincia', 'UBIGEO': 'Ubigeo'}, inplace = True)
mapas[['Ubigeo_Departamento', 'Ubigeo_Provincia', 'Ubigeo']] = mapas[['Ubigeo_Departamento', 'Ubigeo_Provincia', 'Ubigeo']].astype(str)
print(f'Deben haber 1874 filas (distritos al 2021):', mapas.shape)
mapas.head(5)

In [ ]:
# Filtrar las filas donde 'Año' es 2021
dengue_2021 = dengue[dengue['Año'] == 2021]

# Reordenar las columnas según lo solicitado
dengue_2021 = dengue_2021[['Ubigeo_Departamento', 'Ubigeo_Provincia', 'Ubigeo', 'Departamento', 'Provincia', 'Distrito', 'Año', 'Semana', 'Casos']]

dengue_2021.head(5)

In [ ]:
# Definimos una función para sumar los casos.
# Si más del 65% de los valores en el grupo son NA, la suma será NA.
# Si menos del 65% son NA, sumamos los casos ignorando los valores NA.
def sumar_casos(total):
    if total.isna().mean() >= 0.65:
        return pd.NA  # Si más del 65% son NA, devolvemos NA
    else:
        return total.sum()  # Si no, sumamos los valores no NA

# Generamos el nuevo df 'df_2021', agrupando por 'Ubigeo' y aplicando la función sumar_casos
df_2021 = dengue_2021.groupby('Ubigeo', as_index=False).agg(
          Casos_distrital_2021=('Casos', sumar_casos))

# Hacemos un merge para agregar la columna 'Ubigeo' y 'geometry' desde el df 'mapas'
df_2021 = pd.merge(mapas[['Ubigeo_Departamento', 'Ubigeo_Provincia', 'Ubigeo', 'geometry']], df_2021, on='Ubigeo', how='left')

# Hacemos un merge para agregar información desde el df dengue. Debemos indicar drop_duplicates() para no repetir cada fila
df_2021 = pd.merge(df_2021, 
                   dengue_2021[['Departamento', 'Provincia', 'Distrito', 'Ubigeo']].drop_duplicates(), 
                   on = 'Ubigeo', 
                   how = 'left')

# Añadimos una columna Año con valor 2021
df_2021['Año'] = '2021'

# Reordenamos las columnas
df_2021 = df_2021[['Año', 'Ubigeo_Departamento', 'Ubigeo_Provincia', 'Ubigeo', 'Departamento', 'Provincia', 'Distrito', 'Casos_distrital_2021', 'geometry']]

#df_2021.head(20)
print(df_2021.dtypes)
print('\n', f'El n de filas debería ser 1874:', df_2021.shape)

In [ ]:
# Definimos el objeto con las columnas que queremos convertir a string
df_2021_cols = ['Año', 'Ubigeo_Departamento', 'Ubigeo_Provincia', 'Ubigeo']

# Convertimos todas las columnas en F1_col a string
df_2021[df_2021_cols] = df_2021[df_2021_cols].astype('string')

# Convertir la columna a float64 para manejar mejor los NaN
df_2021['Casos_distrital_2021'] = df_2021['Casos_distrital_2021'].astype('float64')

print(df_2021.dtypes, '\n')
df_2021.head(10)

In [ ]:
# Obtenemos los estadísticos descriptivos de la columna 'Casos_distrital_2021'
descriptivos = df_2021['Casos_distrital_2021'].describe()
mediana = df_2021['Casos_distrital_2021'].median()

print(f"Estadísticos descriptivos:\n{descriptivos}\n", f"Mediana: {mediana}", '\n', '\n')

# Generamos un gráfico para visualizar mejor la distribución de los datos
fig, ax = plt.subplots(figsize=(5, 5))
df_2021["Casos_distrital_2021"].hist(bins = 50)

### Figura 1: Casos anuales de dengue a nivel distrital en el 2021

In [ ]:
# Graficamos el mapa especificando un color para los valores NaN (con un tono de gris más claro)
F1 = df_2021.plot(column='Casos_distrital_2021',
                  cmap='YlOrBr', 
                  linewidth= 0.25,
                  edgecolor='#515151',
                  missing_kwds={"color": "#e0e0e0",  # Gris más claro que lightgray
                                #"hatch": "/",       # Cambiamos el hatch a líneas paralelas horizontales
                                "label": "No data"},
                  legend=True,
                  figsize=(15, 15))

# Desactivamos los marcadores del eje izquierdo y debajo del mapa
plt.gca().set_axis_off()

# Añadir etiqueta a la barra de color
cbar = F1.get_figure().get_axes()[1]  # Accedemos al segundo eje, que es la barra de color
cbar.set_ylabel('Casos anuales', fontsize=15, fontweight='bold', rotation=270, labelpad=20)  # Rotamos el texto a 270 grados

# Añadimos título
plt.title('Dengue en el Perú a nivel distrital (2021)', fontsize=18, fontweight='bold', ha='center')

# Añadir el texto en la parte inferior izquierda
plt.text(x=0.01, y=-0.05,
         s="Nota: En gris, los distritos sobre los que no se registró información.\nFuente: Elaboración propia con datos de la DGE-MINSA.",
         transform=plt.gca().transAxes, fontsize=9, ha='left')

F1

#4. Casos anuales de dengue a nivel provincial en el 2021

Primero agregaremos las geometrías del shapefile a nivel provincial.

In [ ]:
# Aplicar dissolve para agrupar por Ubigeo_Provincia y sumar los valores de Casos_distrital_2021
df_f2 = df_2021.dissolve(by='Ubigeo_Provincia', aggfunc={'Ubigeo_Departamento': 'first',
                                                         'Departamento': 'first',
                                                         'Provincia': 'first',
                                                         'Casos_distrital_2021': lambda row: row.sum(min_count=1)}
                        ).reset_index()

# Renombrar la columna Casos_distrital_2021 a Casos_provincial_2021
df_f2.rename(columns={'Casos_distrital_2021': 'Casos_provincial_2021'}, inplace=True)

# Reordenamos las columnas
df_f2 = df_f2[['Ubigeo_Departamento', 'Ubigeo_Provincia', 'Departamento', 'Provincia', 'Casos_provincial_2021', 'geometry']]

print(f'El n de filas debería ser 196:', df_f2.shape, '\n')
df_f2.head(5)

In [ ]:
# Verificamos la distribución de los datos
fig, ax = plt.subplots(figsize=(5, 5))
df_f2["Casos_provincial_2021"].hist(bins = 50)

In [ ]:
# Graficamos el mapa especificando un color para los valores NaN (con un tono de gris más claro)
F2 = df_f2.plot(column='Casos_provincial_2021',
                cmap='YlOrBr', 
                linewidth= 0.25,
                edgecolor='#515151',
                missing_kwds={"color": "#e0e0e0",
                              "hatch": "//",
                              "label": "No data"},
                legend=True,
                figsize=(15, 15))

# Desactivamos los marcadores del eje izquierdo y debajo del mapa
plt.gca().set_axis_off()

# Ocultar los marcadores de valores pero mantener el recuadro
#plt.gca().tick_params(left=False, right=False, labelleft=False, labelbottom=False, bottom=False)

# Añadir etiqueta a la barra de color
cbar = F2.get_figure().get_axes()[1]  # Accedemos al segundo eje, que es la barra de color
cbar.set_ylabel('Casos anuales', fontsize=15, fontweight='bold', rotation=270, labelpad=20)

# Añadimos título
plt.title('Dengue en el Perú a nivel provincial (2021)', fontsize=18, fontweight='bold', ha='center')

# Añadir el texto en la parte inferior izquierda
plt.text(x=0.01, y=-0.05,
         s="Nota: En gris, las provincias sobre las que no se registró información. Las provincias con muy pocos casos pueden tener datos de solo un distrito.\nFuente: Elaboración propia con datos de la DGE-MINSA.",
         transform=plt.gca().transAxes, fontsize=9, ha='left')

F2

#5. Casos anuales de dengue en el Perú a nivel departamental (2015-2021)

Primero, preparamos los datos.

In [ ]:
# Agrupar por Año y Ubigeo_Departamento, sumando los valores de Casos respetando los NaN
df_dep_años = dengue.groupby(['Año', 'Ubigeo_Departamento', 'Departamento'], as_index=False).agg(
              Casos_departamental_anual=('Casos', lambda row: row.sum(min_count=1)))

print(f'El n de filas debería ser 175 (7 años x 25 departamentos), por la F1 y F2 sabemos que faltan departamentos.\nMás adelante verificaremos si faltan años.')
print(f'Dimensiones del df:', df_dep_años.shape)
print('\n')
print(df_dep_años.dtypes)
print('\n')
df_dep_años

In [ ]:
# Convertimos las columnas a los tipos adecuados
df_dep_años['Año'] = df_dep_años['Año'].astype('Int64').astype('string')

Completamos las filas que faltan en df_dep_años para que sean 175 en total (7 años x 25 departamentos).

In [ ]:
# Lista de años como strings
años = [str(a) for a in range(2015, 2022)]  # del 2015 al 2021

# Lista de Ubigeo_Departamento como strings
ubigeo_deps = [f'{i:02d}' for i in range(1, 26)]  # del '01' al '25'

# Bucle para buscar combinaciones faltantes y agregar filas al DataFrame
for año in años:
    for ubigeo_dep in ubigeo_deps:
        # Verificar si la combinación existe
        fila = df_dep_años[(df_dep_años['Año'] == año) & (df_dep_años['Ubigeo_Departamento'] == ubigeo_dep)]
        
        # Si la combinación no existe, agregar una fila con Año y Ubigeo_Departamento (las demás columnas se llenan automáticamente con NaN)
        if fila.empty:
            nueva_fila = pd.DataFrame({'Año': [año], 'Ubigeo_Departamento': [ubigeo_dep]})
            
            # Concatenar la nueva fila al DataFrame original
            df_dep_años = pd.concat([df_dep_años, nueva_fila], ignore_index=True)

# Ordenar el DataFrame por Año y Ubigeo_Departamento
df_dep_años = df_dep_años.sort_values(by=['Año', 'Ubigeo_Departamento'])

# Resetear el índice para reorganizar los números de las filas
df_dep_años.reset_index(drop=True, inplace=True)

# Ver el resultado
df_dep_años

In [ ]:
# Agrupamos las geometrías por departamento usando 'dissolve' para combinar los polígonos de las filas con el mismo valor en 'Ubigeo_Departamento'
# Añadimos 'reset_index()' para que 'Ubigeo_Departamento' vuelva a ser una columna normal en lugar del índice
geo_departamentos = mapas[['Ubigeo_Departamento', 'geometry']].dissolve(by='Ubigeo_Departamento').reset_index()
print(geo_departamentos.dtypes, '\n')
geo_departamentos

In [ ]:
# Hacemos un merge para agregar la columna 'geometry' desde el df 'geo_departamentos'
df_dep_años = pd.merge(geo_departamentos, df_dep_años, on='Ubigeo_Departamento', how='left')

# Reordenamos las columnas
df_dep_años = df_dep_años[['Año', 'Ubigeo_Departamento', 'Departamento', 'Casos_departamental_anual', 'geometry']]

print(df_dep_años.dtypes, '\n')
df_dep_años

In [ ]:
# Convertimos las columnas a los tipos adecuados
df_dep_años['Año'] = df_dep_años['Año'].astype('string')
df_dep_años['Ubigeo_Departamento'] = df_dep_años['Ubigeo_Departamento'].astype('string')
df_dep_años['Casos_departamental_anual'] = df_dep_años['Casos_departamental_anual'].astype(float)
df_dep_años.dtypes

In [ ]:
# Ordenar el DataFrame por Año y Ubigeo_Departamento
df_dep_años = df_dep_años.sort_values(by=['Año', 'Ubigeo_Departamento'])
# Resetear el índice para reorganizar los números de las filas
df_dep_años.reset_index(drop=True, inplace=True)
df_dep_años.head(55)

In [ ]:
# Crear la figura y los subplots
fig, axis = plt.subplots(nrows=3, ncols=3, figsize=(15, 15))

# Lista de años en tu DataFrame
years = df_dep_años['Año'].unique()

# Variable para controlar la iteración en los subplots
idx = 0

# Iterar sobre las filas y columnas de la matriz de subplots
for i in range(3):
    for j in range(3):
        if idx >= len(years):
            axis[i, j].set_axis_off()  # Desactivar ejes vacíos si no hay más años
            continue
        
        ax = axis[i][j]
        
        # Filtrar los datos del año correspondiente
        year = years[idx]
        df_year = df_dep_años[df_dep_años['Año'] == year]
        
        # Plotear los datos del año en la subplot correspondiente
        F3 = df_year.plot(column='Casos_departamental_anual',
                          cmap='YlOrBr',  # Mapa de color
                          edgecolor='#515151',  # Color del borde
                          linewidth= 0.25,
                          legend=True, 
                          missing_kwds={"color": "lightgray",
                                        "hatch": "///",
                                        "label": "No data"},  # Color para valores NA
                          ax=ax)
        
        # Eliminar los marcadores de los ejes
        ax.set_xticks([])  # Eliminar valores en el eje x
        ax.set_yticks([])  # Eliminar valores en el eje y
        
        # Establecer el título del subplot como el año
        ax.set_title(f'Año {year}', fontsize=12, fontweight='bold')
        
        # Incrementar el índice para pasar al siguiente año
        idx += 1

# Título general centrado
fig.suptitle('Casos anuales de dengue en el Perú a nivel departamental (2015-2021)', fontsize=20, fontweight='bold')

# Texto en la parte inferior centrado
fig.text(0.5, 0.01, 
         "Nota: En gris, los departamentos sobre los que no se registró información. Los departamentos con muy pocos casos pueden tener datos de solo un distrito.\nFuente: Elaboración propia con datos de la DGE-MINSA.", 
         ha='center', fontsize=12)

# Ajustar el espacio entre subplots
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()

#6. Casos trimestrales de dengue a nivel departamental en el 2021

In [ ]:
# Crear una copia del df 'dengue_2021' seleccionando las columnas necesarias
df_2021_trimestral = dengue_2021[['Ubigeo_Departamento', 'Departamento', 'Semana', 'Casos']].copy()
print(df_2021_trimestral.dtypes, '\n')
df_2021_trimestral.head(20)

In [ ]:
# Función para asignar el trimestre según el valor de Semana
def asignar_trimestre(semana):
    if 1 <= semana < 14:
        return 'T1'
    elif 14 <= semana < 27:
        return 'T2'
    elif 27 <= semana < 40:
        return 'T3'
    elif 40 <= semana < 53:
        return 'T4'
    else:
        return np.nan  # Por si hay valores inesperados

# Aplicar la función para crear la nueva columna 'Trimestre'
df_2021_trimestral['Trimestre'] = df_2021_trimestral['Semana'].apply(asignar_trimestre)

# Verificar el resultado
df_2021_trimestral.head(5)

df_2021_trimestral.dtypes

df_2021_trimestral = df_2021_trimestral.groupby(['Trimestre', 'Ubigeo_Departamento', 'Departamento'], as_index=False).agg(
                    Casos_departamental_trimestral=('Casos', lambda row: row.sum(min_count=1)))

In [ ]:
# Agrupar por Trimestre, Ubigeo_Departamento y Departamento
df_2021_trimestral = df_2021_trimestral.groupby(['Trimestre', 'Ubigeo_Departamento', 'Departamento'], as_index=False).agg(
                     Casos_departamental_trimestral=('Casos', lambda row: row.sum(min_count=1)))

print(df_2021_trimestral.dtypes, '\n')
print(f'No tiene 100 filas (4 trimestres x 25 departamentos) porque faltan departamentos.', '\n')
print(f'Dimensiones del df:', df_2021_trimestral.shape, '\n')
df_2021_trimestral.head(20)

In [ ]:
# Lista de trimestres como strings
trimestres = ['T1', 'T2', 'T3', 'T4']

# Lista de Ubigeo_Departamento como strings
ubigeo_deps = [f'{i:02d}' for i in range(1, 26)]  # del '01' al '25'

# Bucle para buscar combinaciones faltantes y agregar filas al DataFrame
for trimestre in trimestres:
    for ubigeo_dep in ubigeo_deps:
        # Verificar si la combinación existe
        fila = df_2021_trimestral[(df_2021_trimestral['Trimestre'] == trimestre) & (df_2021_trimestral['Ubigeo_Departamento'] == ubigeo_dep)]
        
        # Si la combinación no existe, agregar una fila con Trimestre y Ubigeo_Departamento (las demás columnas se llenan automáticamente con NaN)
        if fila.empty:
            nueva_fila = pd.DataFrame({'Trimestre': [trimestre], 'Ubigeo_Departamento': [ubigeo_dep]})
            
            # Concatenar la nueva fila al DataFrame original
            df_2021_trimestral = pd.concat([df_2021_trimestral, nueva_fila], ignore_index=True)

# Ordenar el DataFrame por Trimestre y Ubigeo_Departamento
df_2021_trimestral = df_2021_trimestral.sort_values(by=['Trimestre', 'Ubigeo_Departamento'])

# Resetear el índice para reorganizar los números de las filas
df_2021_trimestral.reset_index(drop=True, inplace=True)

# Ver el resultado
df_2021_trimestral

In [ ]:
# Hacemos un merge para agregar los departamentos que faltan y la columna 'geometry' desde el df 'geo_departamentos'
df_2021_trimestral = pd.merge(geo_departamentos, df_2021_trimestral, on='Ubigeo_Departamento', how='left')

# Reordenamos las columnas
df_2021_trimestral = df_2021_trimestral[['Trimestre', 'Ubigeo_Departamento', 'Departamento', 'Casos_departamental_trimestral', 'geometry']]

print(df_2021_trimestral.dtypes, '\n')
print(f'Debe tener 100 filas:', df_2021_trimestral.shape, '\n')
df_2021_trimestral.head(10)

In [ ]:
# Convertimos las columnas a los tipos adecuados
df_2021_trimestral['Trimestre'] = df_2021_trimestral['Trimestre'].astype('string')
df_2021_trimestral['Ubigeo_Departamento'] = df_2021_trimestral['Ubigeo_Departamento'].astype('string')
df_2021_trimestral.dtypes

Generamos el gráfico para cada trimestre del 2021

In [ ]:
# Crear la figura y los subplots
fig, axis = plt.subplots(nrows=2, ncols=2, figsize=(15, 15))

# Lista de trimestres en tu DataFrame
trimestres = df_2021_trimestral['Trimestre'].unique()

# Variable para controlar la iteración en los subplots
idx = 0

# Iterar sobre las filas y columnas de la matriz de subplots
for i in range(2):
    for j in range(2):
        if idx >= len(trimestres):
            axis[i, j].set_axis_off()  # Desactivar ejes vacíos si no hay más trimestres
            continue
        
        ax = axis[i][j]
        
        # Filtrar los datos del trimestre correspondiente
        trimestre = trimestres[idx]
        df_trimestre = df_2021_trimestral[df_2021_trimestral['Trimestre'] == trimestre]
        
        # Plotear los datos del trimestre en el subplot correspondiente
        F4 = df_trimestre.plot(column='Casos_departamental_trimestral',
                               cmap='YlOrBr',  # Mapa de color
                               edgecolor='#515151',  # Color del borde
                               linewidth= 0.25,
                               legend=True, 
                               missing_kwds={"color": "lightgray",
                                             "hatch": "///",
                                             "label": "Sin datos"},  # Color para valores NA
                               ax=ax,
                               scheme='quantiles',  # Definir esquema categórico
                               classification_kwds={'bins': 5},
                               legend_kwds={'loc': 'upper left',
                                            'bbox_to_anchor': (1.05, 1),
                                            'edgecolor': 'black',
                                            'title': 'Casos de dengue',
                                            'fmt': '{:.0f}'})  # Esto elimina los decimales en la leyenda)
        
        # Eliminar los marcadores de los ejes
        ax.set_xticks([])  # Eliminar valores en el eje x
        ax.set_yticks([])  # Eliminar valores en el eje y
        
        # Establecer el título del subplot como el trimestre
        ax.set_title(f'Trimestre {trimestre.replace("T", "")}', fontsize=12, fontweight='bold')
        
        # Incrementar el índice para pasar al siguiente trimestre
        idx += 1

# Título general centrado
fig.suptitle('Casos trimestrales de dengue en el Perú a nivel departamental (2021)', fontsize=20, fontweight='bold')

# Texto en la parte inferior centrado
fig.text(0.5, 0.01, 
         "Nota: En gris, los departamentos sobre los que no se registró información. Los departamentos con muy pocos casos pueden tener datos de solo un distrito.\nFuente: Elaboración propia con datos de la DGE-MINSA.", 
         ha='center', fontsize=12)


# Ajustar el margen izquierdo para mover los subplots más a la izquierda
plt.subplots_adjust(left=0.04, top=0.9, bottom=0.07)  # Puedes ajustar este valor para mover más o menos
                    #hspace: controla el espacio vertical entre subplots.
                    #wspace: controla el espacio horizontal entre subplots.
                    #left, right, top, bottom: controlan el margen exterior (espacio entre los bordes de la figura y los subplots).

# Ajustar el espacio entre subplots
#plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.show()